<a href="https://colab.research.google.com/github/farahiqbal321/AI-BCU-Hackathon/blob/main/AI_BCU_Hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get update -qq
!apt-get install -y zstd
!pip install -q pandas openpyxl ddgs scikit-learn ollama

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 105 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (2,921 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 

In [2]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
import subprocess
import time

ollama_process = subprocess.Popen(["ollama", "serve"])
time.sleep(8)

print("Ollama server started")

Ollama server started


In [4]:
!ollama pull qwen2.5:3b
!ollama pull llama3.2:3b
!ollama pull phi3:mini

In [5]:
import re
import time
from pathlib import Path
from typing import Dict, List
from collections import Counter

import pandas as pd
from ddgs import DDGS
import ollama

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
QUESTIONS_FILE = "questions_100.csv"
OUTPUT_FILE = "EvidenceAI_submission.csv"
DEBUG_FILE = "EvidenceAI_debug_log.csv"

OLLAMA_MODELS = [
    "qwen2.5:3b",
    "llama3.2:3b",
    "phi3:mini"
]

ENABLE_WEB_SEARCH = True
MAX_RESULTS_PER_QUERY = 6
SEARCH_SLEEP_SECONDS = 0.5
TOP_K_EVIDENCE = 5

ALLOWED_ANSWERS = {"A", "B", "C", "D", "E", "Unknown"}

In [7]:
from google.colab import files

uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))

Saving questions_100.csv to questions_100.csv
Uploaded files: ['questions_100.csv']


In [8]:
def load_questions(path: str) -> pd.DataFrame:
    file_path = Path(path)

    if not file_path.exists():
        raise FileNotFoundError(f"Could not find {path}. Please upload questions_100.csv.")

    questions = pd.read_csv(file_path)

    required_columns = {"question_no", "question"}
    missing = required_columns - set(questions.columns)

    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    return questions


questions = load_questions(QUESTIONS_FILE)
questions.head()

,question_no,question,A,B,C,D,E
0,1,What are the core capabilities of United State...,Electronic warfare and intelligence analysis.,"Direct action, special reconnaissance and fore...",Airborne operations and cyber warfare.,Maritime interdiction and unconventional warfare.,Counterintelligence and emergency medical resp...
1,2,In which war did General Sir George Anson serv...,Crimean War,War of 1812,American Revolutionary War,Napoleonic Wars,French Revolution
2,3,What was P. Padmarajan's contribution to Malay...,P. Padmarajan was a noted film critic who revo...,"P. Padmarajan was an Indian film maker, screen...",P. Padmarajan was a pioneer in introducing adv...,P. Padmarajan was a renowned cinematographer k...,P. Padmarajan popularized the use of complex n...
3,4,What is the length of U.S. Route 11 in Georgia?,11 miles (17.7 km),36.7 miles (58.9 km),58 miles (93.3 km),22.8 miles (36.7 km),8 miles (12.9 km)
4,5,How is Edmond Malone best known for his contri...,Edmond Malone is best known for his extensive ...,Edmond Malone is best known for his comprehens...,Edmond Malone is best known for his groundbrea...,Edmond Malone is best known for his meticulous...,Edmond Malone is best known for his pioneering...


In [9]:
def get_options(row: pd.Series) -> List[str]:
    options = []

    for option in ["A", "B", "C", "D", "E"]:
        value = row.get(option, "")

        if pd.notna(value) and str(value).strip():
            options.append(f"{option}. {str(value).strip()}")

    return options

In [10]:
def build_search_queries(row: pd.Series) -> List[str]:
    question = str(row.get("question", "")).strip()

    option_texts = []

    for option in ["A", "B", "C", "D", "E"]:
        value = row.get(option, "")

        if pd.notna(value) and str(value).strip():
            option_texts.append(str(value).strip())

    queries = [
        question,
        question + " " + " ".join(option_texts),
        question + " wikipedia",
        question + " facts"
    ]

    cleaned_queries = []

    for query in queries:
        query = re.sub(r"\s+", " ", query).strip()
        query = query[:300]

        if query and query not in cleaned_queries:
            cleaned_queries.append(query)

    return cleaned_queries

In [11]:
def search_duckduckgo(query: str, max_results: int = MAX_RESULTS_PER_QUERY) -> List[Dict[str, str]]:
    if not ENABLE_WEB_SEARCH:
        return []

    evidence = []

    try:
        with DDGS() as ddgs:
            results = ddgs.text(query, max_results=max_results)

            for item in results:
                evidence.append({
                    "title": item.get("title", ""),
                    "snippet": item.get("body", ""),
                    "url": item.get("href", ""),
                })

    except Exception as error:
        print(f"[Warning] DuckDuckGo search failed: {error}")

    time.sleep(SEARCH_SLEEP_SECONDS)

    return evidence

In [12]:
def retrieve_evidence(row: pd.Series) -> List[Dict[str, str]]:
    all_evidence = []
    seen_urls = set()

    for query in build_search_queries(row):
        results = search_duckduckgo(query)

        for item in results:
            url = item.get("url", "")

            if url and url not in seen_urls:
                seen_urls.add(url)
                all_evidence.append(item)

    return all_evidence

In [13]:
def rank_evidence(row: pd.Series, evidence: List[Dict[str, str]], top_k: int = TOP_K_EVIDENCE) -> List[Dict[str, str]]:
    if not evidence:
        return []

    question = str(row.get("question", "")).strip()
    options = " ".join(get_options(row))

    query_text = question + " " + options

    evidence_texts = [
        f"{item.get('title', '')} {item.get('snippet', '')}"
        for item in evidence
    ]

    documents = [query_text] + evidence_texts

    try:
        vectorizer = TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2),
            max_features=5000
        )

        tfidf_matrix = vectorizer.fit_transform(documents)

        query_vector = tfidf_matrix[0:1]
        evidence_vectors = tfidf_matrix[1:]

        scores = cosine_similarity(query_vector, evidence_vectors)[0]

        ranked = []

        for score, item in zip(scores, evidence):
            item = item.copy()
            item["score"] = float(score)
            ranked.append(item)

        ranked = sorted(ranked, key=lambda x: x["score"], reverse=True)

        return ranked[:top_k]

    except Exception as error:
        print(f"[Warning] TF-IDF ranking failed: {error}")
        return evidence[:top_k]

In [14]:
def build_prompt(row: pd.Series, ranked_evidence: List[Dict[str, str]]) -> str:
    question = str(row.get("question", "")).strip()
    options = get_options(row)

    evidence_text = "\n\n".join([
        f"Evidence {i}:\nTitle: {item.get('title', '')}\nSnippet: {item.get('snippet', '')}\nURL: {item.get('url', '')}"
        for i, item in enumerate(ranked_evidence, start=1)
    ])

    prompt = f"""
You are a careful multiple-choice question answering system.

Your task:
- Read the question.
- Read the answer options.
- Use the evidence where useful.
- Choose the best supported answer.
- Return only one of: A, B, C, D, E, Unknown.

Do not explain your answer.
Do not write a sentence.
Only output the final answer letter.

Question:
{question}

Options:
{chr(10).join(options)}

Evidence:
{evidence_text}

Final answer:
""".strip()

    return prompt

In [15]:
def clean_answer(answer: str) -> str:
    answer = str(answer).strip()

    match = re.search(r"\b(A|B|C|D|E|Unknown)\b", answer, re.IGNORECASE)

    if match:
        result = match.group(1)

        if result.lower() == "unknown":
            return "Unknown"

        return result.upper()

    first_char = answer.upper()[:1]

    if first_char in {"A", "B", "C", "D", "E"}:
        return first_char

    return "Unknown"

In [16]:
def generate_answer(prompt: str, model_name: str) -> str:
    try:
        response = ollama.chat(
            model=model_name,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            options={
                "temperature": 0,
                "num_predict": 20
            }
        )

        output = response["message"]["content"]
        return clean_answer(output)

    except Exception as error:
        print(f"[Warning] {model_name} failed: {error}")
        return "Unknown"

In [17]:
def majority_vote(answers: List[str]) -> tuple[str, float]:
    valid_answers = [a for a in answers if a in ALLOWED_ANSWERS]

    if not valid_answers:
        return "Unknown", 0.0

    counts = Counter(valid_answers)
    final_answer, vote_count = counts.most_common(1)[0]
    confidence = vote_count / len(valid_answers)

    return final_answer, confidence

In [18]:
def answer_question(row: pd.Series) -> Dict[str, str]:
    evidence = retrieve_evidence(row)
    ranked_evidence = rank_evidence(row, evidence)
    prompt = build_prompt(row, ranked_evidence)

    model_answers = []

    for model_name in OLLAMA_MODELS:
        answer = generate_answer(prompt, model_name)
        model_answers.append(answer)

    final_answer, confidence = majority_vote(model_answers)

    evidence_summary = " | ".join([
        f"{item.get('title', '')}: {item.get('snippet', '')}"
        for item in ranked_evidence
    ])

    return {
        "question_no": row.get("question_no"),
        "answer": final_answer,
        "confidence": confidence,
        "votes": ",".join(model_answers),
        "top_evidence": evidence_summary
    }

In [19]:
def run_pipeline(
    questions: pd.DataFrame,
    output_file: str,
    debug_file: str,
    limit: int | None = None
) -> tuple[pd.DataFrame, pd.DataFrame]:

    if limit is not None:
        questions = questions.head(limit)

    results = []

    for index, row in questions.iterrows():
        question_no = row.get("question_no", index + 1)

        print(f"Answering question {question_no}...")

        result = answer_question(row)
        results.append(result)

        print(
            f"Answer: {result['answer']} | "
            f"Confidence: {result['confidence']} | "
            f"Votes: {result['votes']}"
        )

    debug_df = pd.DataFrame(results)
    debug_df.to_csv(debug_file, index=False)

    submission = debug_df[["question_no", "answer"]].copy()

    invalid_answers = set(submission["answer"]) - ALLOWED_ANSWERS

    if invalid_answers:
        raise ValueError(f"Invalid answers found: {invalid_answers}")

    submission.to_csv(output_file, index=False)

    print(f"\nSaved final submission to: {output_file}")
    print(f"Saved debug log to: {debug_file}")

    return submission, debug_df

In [20]:
submission, debug_df = run_pipeline(
    questions,
    OUTPUT_FILE,
    DEBUG_FILE,
    limit=5
)

submission.head()

Answering question 1...
Answer: B | Confidence: 1.0 | Votes: B,B,B
Answering question 2...
Answer: D | Confidence: 1.0 | Votes: D,D,D
Answering question 3...
Answer: B | Confidence: 1.0 | Votes: B,B,B
Answering question 4...
Answer: D | Confidence: 0.6666666666666666 | Votes: D,B,D
Answering question 5...
Answer: D | Confidence: 0.6666666666666666 | Votes: A,D,D

Saved final submission to: EvidenceAI_submission.csv
Saved debug log to: EvidenceAI_debug_log.csv


,question_no,answer
0,1,B
1,2,D
2,3,B
3,4,D
4,5,D


In [21]:
submission, debug_df = run_pipeline(
    questions,
    OUTPUT_FILE,
    DEBUG_FILE,
    limit=None
)

submission.head()

Answering question 1...
Answer: B | Confidence: 1.0 | Votes: B,B,B
Answering question 2...
Answer: D | Confidence: 1.0 | Votes: D,D,D
Answering question 3...
Answer: B | Confidence: 1.0 | Votes: B,B,B
Answering question 4...
Answer: D | Confidence: 1.0 | Votes: D,D,D
Answering question 5...
Answer: D | Confidence: 1.0 | Votes: D,D,D
Answering question 6...
Answer: A | Confidence: 1.0 | Votes: A,A,A
Answering question 7...
Answer: E | Confidence: 0.6666666666666666 | Votes: D,E,E
Answering question 8...
Answer: A | Confidence: 0.3333333333333333 | Votes: A,B,C
Answering question 9...
Answer: C | Confidence: 1.0 | Votes: C,C,C
Answering question 10...
Answer: B | Confidence: 0.6666666666666666 | Votes: B,A,B
Answering question 11...
Answer: A | Confidence: 0.6666666666666666 | Votes: A,A,B
Answering question 12...
Answer: B | Confidence: 1.0 | Votes: B,B,B
Answering question 13...
Answer: C | Confidence: 1.0 | Votes: C,C,C
Answering question 14...
Answer: B | Confidence: 1.0 | Votes: B,B

,question_no,answer
0,1,B
1,2,D
2,3,B
3,4,D
4,5,D
